# Cape and Islands: answers to the 23 shared Objectives
## Dataset-only feasibility, numerical answers where supported, and uniform chart definitions

This follows **Objectives Q1–Q23**, grouped by scope, in the team’s shared question document. It does not use the later final shortlist. Question wording is embedded; no numerical result from Berkshire or any external source is imported. Cape and Islands is treated as one jurisdiction.

Each question has a status, an answer or missing-data reason, and a chart where supported. **Answerable** means the literal conditional quantity is observable. **Partial** means only a specified component/proxy is observable. **Not answerable** means a required input is absent. **Insufficient data** means the relevant completeness rule fails. Nearby quantities are labeled companions, not substituted silently.

No verified filing date exists, so filing-year trends and processing-time goals cannot be copied from Berkshire. Missing values remain a separate category throughout. All code and results are included here; original workbooks are read-only.

## Scope, units, and limits shared by both Cape notebooks

- **Only local inputs:** `charges filed.xlsx`, `charges disposed.xlsx`, `bail_data.xlsx`. No external statistics, population estimates, crime lists, other-county records, or policy dates are loaded. The 23 Objective wordings are embedded as configuration.
- **No filing date exists.** Neither offense date nor disposition date substitutes for filing date. The leading two digits of IDs matching `YY-MM-number` supply an **unverified case-ID-year proxy**, used only in labeled exploratory charts. Records without a usable ID year stay in an explicit Missing proxy-year category in full-file audits. No month/day of filing is invented.
- **Study populations:** disposition rows and bail events dated 2017–2024 have observed date filters. “Filing extract: ID-year cohort 2017–2024” uses the proxy and cannot be compared directly to Berkshire’s verified initiation-year population. Full-file tables are labeled as such and retain unknown proxy years. Old offense dates can be legitimate; malformed centuries/future dates are flagged rather than guessed.
- **Missing is a category, not zero.** Source blanks become `Missing`; explicit `U` codes become `Unknown (U)`; unmapped codes remain separate; conflicting values are flagged. Missing monetary amounts never become $0. Invalid dates remain distinguishable from source blanks in the audit.
- **Repeated rows are retained.** The export has no charge sequence ID. An identical row may be a separate count of the same offense. The repeated total-charge field is compared against row counts. Deduplication is only a labeled sensitivity, not automatic cleaning.
- **One case-defendant unit:** internal case number + defendant ID. Docket numbers are audited; an internal ID is a documented proxy for the team’s docket-per-defendant rule. Bail lacks defendant ID, so only case IDs with one observed defendant are linked. Aggregate first, then merge one-to-one; no charge-to-event Cartesian joins.
- **Source class:** only `F`, `M+`, `M-`, `CM` are recognized. Blank and nonstandard codes such as `PD`/`CATP` are not filled or mapped into felony. A case with known felony has a confirmed felony marker; otherwise missing/unmapped charge classes prevent a definite highest-class assignment. Representative charge is a deterministic highest-known-source-class proxy, not a validated statutory seriousness ranking.
- **Criminal scope:** cases/rows with at least one recognized F/M+/M− charge are a selected known-criminal population. Unknown/unmapped-only cases are not declared civil. Civil responsibility results and juvenile adjudication are separate from criminal conviction/acquittal.
- **Case outcomes:** observed closed cases with latest valid recorded disposition in 2017–2024, using all supplied disposed rows to detect later dispositions. Current status is not a historical snapshot, and complete case closure is not proven. Provisional hierarchy: trial conviction > guilty plea > juvenile adjudication > CWOF > recorded pretrial probation > acquittal marker > nolle prosequi > dismissal > civil result > procedural result. Unresolved charges override the hierarchy except an explicit top trial-conviction marker. CWOF is not a conviction or automatically a diversion program.
- **Coverage rule:** dependent rate claims are withheld when >20% of the relevant population is missing/unresolved; comparative cells below 30 are suppressed. Selected observed-record diagnostics are not county-wide estimates. Unknowns remain visible in denominators.
- **Sentences and bail:** `F` sentence code means **Filed**, not Fine. `SS` means Suspended, not proof of probation. The duplicated “Imposed MAX Term Unit” columns are not guessed to mean MIN/MAX; duration uses uniquely labeled Actual MAX fields. Set/requested bail is not money paid, received, revenue, forfeiture, or inability to post. `BAIL` is the recorded Bail label, not verified cash-only or surety.
- **No causal attribution:** observed demographic, judge, court, agency, charge, or bail differences do not establish discrimination, official performance, or causal effects. Charges are allegations. No age, full prior record, separate ethnicity, complete referrals, payment history, or detention-duration history is supplied.

**Reproduce:** install `requirements_cape.txt`, open in Jupyter/VS Code, restart and Run All from this folder or its parent. No network is needed. Outputs are embedded and exported to the named output folder. Original workbooks are never written; SHA-256 hashes are checked at the end.

In [ ]:
from pathlib import Path
import calendar
import hashlib
import json
import platform
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from IPython.display import display, Markdown
import openpyxl

warnings.filterwarnings('ignore', category=FutureWarning)
START = pd.Timestamp('2017-01-01')
END = pd.Timestamp('2024-12-31')
FOLLOWUP_END = pd.Timestamp('2025-12-31')
MIN_CELL = 30
TOP_N = 12
DATA_DIR = Path.cwd()
if not (DATA_DIR / 'charges filed.xlsx').exists():
    DATA_DIR = DATA_DIR / 'cape and island'
assert (DATA_DIR / 'charges filed.xlsx').exists(), 'Run from the Berkshire folder or its parent.'
OUT = DATA_DIR / 'cape_objectives_23_outputs'
(OUT / 'figures').mkdir(parents=True, exist_ok=True)
(OUT / 'tables').mkdir(exist_ok=True)
TABLES, FINDINGS, FIGURES = {}, [], []

COLORS = ['#205072','#2c8c86','#e0a13a','#b75453','#7f7099','#69a5cc','#8f9779','#c388a3','#777777','#b7c2cb']
plt.rcParams.update({'figure.dpi':110, 'savefig.dpi':150, 'font.family':'DejaVu Sans',
                     'font.size':10, 'axes.titlesize':14, 'axes.labelsize':10,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.prop_cycle':plt.cycler(color=COLORS)})
pd.set_option('display.max_columns', 16)
pd.set_option('display.max_colwidth', 65)

def note(text):
    FINDINGS.append(text)
    display(Markdown('**Readout.** ' + text))

def table(name, frame, rows=20, digits=2):
    frame = frame.to_frame() if isinstance(frame, pd.Series) else frame.copy()
    TABLES[name] = frame
    frame.to_csv(OUT / 'tables' / f'{name}.csv', index=True)
    display(frame.head(rows).round(digits))
    if len(frame) > rows:
        display(Markdown(f'Showing {rows:,} of {len(frame):,} rows. Complete table: `eda_outputs/tables/{name}.csv`.'))
    return frame

def finish(fig, name, caption):
    fig.text(.01, .01, caption, fontsize=8, color='#4f5c66', wrap=True)
    fig.tight_layout(rect=(0,.07,1,.98))
    fig.savefig(OUT / 'figures' / f'{name}.png', bbox_inches='tight')
    FIGURES.append(name)
    plt.show()
    plt.close(fig)

def barh(series, title, xlabel='Recorded charge rows', color=None):
    series = series.sort_values()
    fig, ax = plt.subplots(figsize=(11,max(4, .36*len(series)+1.3)))
    ax.barh([str(x) for x in series.index], series.values, color=color or COLORS[0])
    ax.set(title=title, xlabel=xlabel)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:,.0f}'))
    for i, value in enumerate(series.values):
        ax.text(value, i, f' {value:,.0f}', va='center', fontsize=9)
    ax.margins(x=.18)
    return fig, ax

def stacked(frame, title, percent=False):
    frame = frame.copy()
    if percent:
        frame = frame.div(frame.sum(axis=1).replace(0,np.nan), axis=0)*100
    fig, ax = plt.subplots(figsize=(12,6))
    frame.plot.bar(stacked=True, ax=ax, color=COLORS[:len(frame.columns)], width=.8)
    ax.set(title=title, ylabel='Share within group (%)' if percent else 'Recorded charge rows', xlabel='')
    ax.tick_params(axis='x', labelrotation=35)
    if percent: ax.set_ylim(0,100)
    ax.legend(title='', bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
    return fig, ax

def heatmap(frame, title, percent=False, annotate=False, cmap='Blues'):
    frame=frame.dropna(how='all').dropna(axis=1,how='all')
    if frame.empty:
        fig,ax=plt.subplots(figsize=(10,4))
        ax.text(.5,.5,'Insufficient data after the reporting threshold',ha='center',va='center',transform=ax.transAxes)
        ax.set_title(title);ax.set_axis_off()
        return fig,ax
    fig, ax = plt.subplots(figsize=(max(9,.72*len(frame.columns)),max(4,.4*len(frame)+1.8)))
    arr = frame.to_numpy(dtype=float)
    im = ax.imshow(np.ma.masked_invalid(arr), aspect='auto', cmap=cmap,
                   vmin=0)
    ax.set_xticks(range(len(frame.columns)), [str(c) for c in frame.columns], rotation=40, ha='right')
    ax.set_yticks(range(len(frame)), [str(i) for i in frame.index])
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label='Percent' if percent else 'Charge rows')
    if annotate:
        midpoint = np.nanmax(arr)/2
        for i in range(len(frame)):
            for j in range(len(frame.columns)):
                value=arr[i,j]
                ax.text(j,i, '—' if np.isnan(value) else f'{value:.0f}', ha='center',va='center',
                        color='white' if value>midpoint else '#263746',fontsize=8)
    return fig, ax

def in_window(series, end=END):
    return series.between(START,end)

def parse_date(series):
    # Keep raw columns separately. Malformed centuries are flagged, not guessed.
    parsed = pd.to_datetime(series, format='mixed', errors='coerce')
    years = parsed.dt.year
    return parsed.where(years.between(1900,2026)).astype('datetime64[ns]')

def normalize_id(series):
    return series.map(lambda x: str(int(x)) if isinstance(x,(float,np.floating)) and pd.notna(x) and x.is_integer()
                      else str(x).strip() if pd.notna(x) else pd.NA).astype('string')

def load_workbook(path):
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    frames, header = [], None
    for sheet in wb:
        values = sheet.values
        first = next(values)
        if header is None:
            header = [str(v).strip() for v in first]
            rows, first_excel_row = list(values), 2
        elif [str(v).strip() for v in first] == header:
            rows, first_excel_row = list(values), 2
        else:
            rows, first_excel_row = [first] + list(values), 1
        part = pd.DataFrame(rows, columns=header)
        part['source_sheet'] = sheet.title
        part['source_row'] = np.arange(first_excel_row, first_excel_row+len(part))
        frames.append(part)
    wb.close()
    frame = pd.concat(frames, ignore_index=True)
    for column in header:
        frame[column] = frame[column].map(lambda x: x.strip() if isinstance(x,str) else x)
        frame[column] = frame[column].replace(r'^\s*$', pd.NA, regex=True)
    return frame

In [ ]:
AS_OF=pd.Timestamp('2026-11-05')
def text_category(s):
    return s.fillna('Missing').astype(str)

def parse_date(s):
    p=pd.to_datetime(s,format='mixed',errors='coerce')
    return p.where(p.ge(pd.Timestamp('1900-01-01'))&p.le(AS_OF)).astype('datetime64[ns]')

def load_cape(path):
    wb=openpyxl.load_workbook(path,read_only=True,data_only=True)
    parts=[];headers=[]
    for sheet in wb:
        values=sheet.values;raw_header=list(next(values));seen={};names=[]
        for i,h in enumerate(raw_header):
            name=str(h).strip() if h is not None else f'Unnamed_{i+1}'
            seen[name]=seen.get(name,0)+1
            names.append(name if seen[name]==1 else f'{name}__{seen[name]}')
            headers.append({'file':path.name,'sheet':sheet.title,'position':i+1,'trimmed_header':name,
                            'analysis_name':names[-1],'duplicate_label':seen[name]>1})
        part=pd.DataFrame(list(values),columns=names)
        for c in names:
            part[c]=part[c].map(lambda x:x.strip() if isinstance(x,str) else x)
            part[c]=part[c].replace(r'^\s*$',pd.NA,regex=True)
        part['exact_repeated_export']=part.duplicated(names,keep='first')
        part['source_sheet']=sheet.title;part['source_row']=np.arange(2,len(part)+2)
        parts.append(part)
    wb.close()
    return pd.concat(parts,ignore_index=True),pd.DataFrame(headers)

paths={'filed':DATA_DIR/'charges filed.xlsx','disposed':DATA_DIR/'charges disposed.xlsx','bail':DATA_DIR/'bail_data.xlsx'}
hashes={k:hashlib.sha256(v.read_bytes()).hexdigest() for k,v in paths.items()}
loaded={k:load_cape(v) for k,v in paths.items()}
raw={k:v[0] for k,v in loaded.items()}
headers=pd.concat([v[1] for v in loaded.values()],ignore_index=True)
table('source_header_audit',headers,rows=12)
inventory=[]
for name,frame in raw.items():
    source_cols=[c for c in frame if c not in ['source_row','source_sheet','exact_repeated_export']]
    inventory.append({'extract':name,'rows':len(frame),'columns':len(source_cols),
      'all_blank_columns':sum(frame[c].isna().all() for c in source_cols),
      'repeated_identical_rows_after_first':int(frame.exact_repeated_export.sum())})
table('source_inventory',pd.DataFrame(inventory))
blank_fields=[]
for name,frame in raw.items():
    for column in frame:
        if frame[column].isna().all():blank_fields.append({'extract':name,'all_blank_field':column})
table('entirely_blank_source_fields',pd.DataFrame(blank_fields))
assert [len(raw[x]) for x in ['filed','disposed','bail']]==[130242,115763,40274]

charge_rename={'Docket Number':'docket','Case Number':'case_id','Defendant ID':'person_id',
 '# of Charges on Case':'declared_charge_count','Offense Date':'offense_raw','Disposition Date':'disposition_raw',
 'Charge Code':'charge_code','Charge Code Description':'charge_description','Charge Category':'class_raw',
 'Court Location Description':'court','Description':'agency','Description__2':'case_status',
 'Disposition':'disposition_code','Disposition Description':'disposition_description',
 'Disposition Event Description':'event_description','Dispostion Comment':'disposition_comment',
 'Judge Name':'judge','Defendant Sex':'sex_code','Race':'race_code',
 'Imposed Sent. Type Code':'sentence_code','Imposed Sent. Type Desc.':'sentence_description',
 'Imposed Sent. Date':'sentence_date_raw','Pros. Rec. Sent. Date':'recommended_date_raw'}
F=raw['filed'].rename(columns=charge_rename).copy()
D=raw['disposed'].rename(columns=charge_rename).copy()
B=raw['bail'].rename(columns={'Case Number':'case_id','Status':'bail_code','Description':'bail_description',
 'Bail Amount Requested':'requested','Bail Amount':'set_amount','Date':'bail_raw',
 'Defendant Sex':'sex_code','Race':'race_code','List of Charges on Case':'charge_list',
 'List of Charges on Case (Desc)':'charge_list_description'}).copy()
race_map={'W':'White','B':'Black','H':'Hispanic (source race field)','A':'Asian','U':'Unknown (U)'}
sex_map={'M':'Male','F':'Female','U':'Unknown (U)'}
date_audit=[];quarantine=[]
for name,frame in [('filed',F),('disposed',D),('bail',B)]:
    frame['case_id']=normalize_id(frame.case_id)
    if 'person_id' in frame:
        frame['person_id']=normalize_id(frame.person_id)
        frame['case_person']=frame.case_id+' | '+frame.person_id.fillna('Missing person ID')
        frame['charge_code']=normalize_id(frame.charge_code)
        frame['declared_charge_count']=pd.to_numeric(frame.declared_charge_count,errors='coerce')
        frame['class_raw']=frame.class_raw.astype('string')
        frame['source_class']=frame.class_raw.fillna('Missing')
        frame['class_plot']=frame.source_class.where(frame.source_class.isin(['F','M+','M-','CM','Missing']),'Unmapped code')
        frame['class_unmapped']=frame.class_raw.notna()&~frame.class_raw.isin(['F','M+','M-','CM'])
        frame['known_criminal']=frame.class_raw.isin(['F','M+','M-'])
        invalid_sex=frame.sex_code.notna()&~frame.sex_code.isin(['M','F','U'])
        frame['severe_shift']=frame.sex_code.eq('Closed').fillna(False)
        frame['unattached_row']=frame.person_id.isna()&frame.charge_code.isna()&frame.charge_description.isna()
        for _,row in frame.loc[frame.severe_shift].iterrows():
            quarantine.append({'extract':name,'sheet':row.source_sheet,'excel_row':int(row.source_row),
              'reason':'Closed appears in sex field; shifted demographic/status pattern'})
        for _,row in frame.loc[frame.unattached_row].iterrows():
            quarantine.append({'extract':name,'sheet':row.source_sheet,'excel_row':int(row.source_row),
              'reason':'unattached row: defendant, charge code and description all missing'})
    else:frame['severe_shift']=False
    frame['race']=frame.race_code.map(lambda x:'Missing' if pd.isna(x) else race_map.get(x,f'Unmapped code: {x}'))
    frame['sex']=frame.sex_code.map(lambda x:'Missing' if pd.isna(x) else sex_map.get(x,f'Unmapped code: {x}'))
    frame['id_year_proxy']=pd.to_numeric(frame.case_id.str.extract(r'^(\d{2})-\d{2}-\d+$')[0],errors='coerce')+2000
    frame['id_year_category']=frame.id_year_proxy.map(lambda x:'Missing proxy year' if pd.isna(x) else str(int(x)))
    for source,target in [('offense_raw','offense_date'),('disposition_raw','disposition_date'),
                           ('sentence_date_raw','sentence_date'),('recommended_date_raw','recommended_date'),('bail_raw','bail_date')]:
        if source in frame:
            parsed=pd.to_datetime(frame[source],format='mixed',errors='coerce')
            frame[target]=parse_date(frame[source])
            frame[target+'_quality']=np.select([frame[source].isna(),frame[target].isna()],['Missing','Invalid / out of bounds'],default='Valid')
            date_audit.append({'extract':name,'field':source,'rows':len(frame),
              'source_blank':int(frame[source].isna().sum()),'invalid_nonblank':int((frame[source].notna()&frame[target].isna()).sum()),
              'raw_min':str(parsed.min()),'raw_max':str(parsed.max()),
              'before_1900':int(parsed.lt(pd.Timestamp('1900-01-01')).sum()),'after_as_of':int(parsed.gt(AS_OF).sum())})

quarantine=pd.DataFrame(quarantine)
table('severely_shifted_row_coordinates',quarantine)
table('date_quality_audit',pd.DataFrame(date_audit),rows=20)
f_all=F.loc[~F.severe_shift&~F.unattached_row].copy();d_all=D.loc[~D.severe_shift&~D.unattached_row].copy();b_all=B.copy()
f=f_all.loc[f_all.id_year_proxy.between(2017,2024)].copy()
d=d_all.loc[in_window(d_all.disposition_date)].copy()
b=b_all.loc[in_window(b_all.bail_date)].copy()
for frame in [b_all,b]:
    for c in ['requested','set_amount']:frame[c]=pd.to_numeric(frame[c],errors='coerce')
canonical=f_all.groupby('charge_code').charge_description.agg(lambda s:s.dropna().mode().iloc[0] if s.notna().any() else 'Missing description')
labels={c:f'{c} · {str(t)[:55]}' for c,t in canonical.items()}
table('analysis_populations',pd.DataFrame({'rows':[len(f_all),len(f),len(d),len(b)],
 'case_ids':[f_all.case_id.nunique(),f.case_id.nunique(),d.case_id.nunique(),b.case_id.nunique()]},
 index=['Full clean filing extract','Filing ID-year proxy cohort 2017–2024','Disposition-date rows 2017–2024','Bail-date events 2017–2024']))
coverage=[]
for name,frame,cols in [('Filing extract: full',f_all,['race_code','sex_code','class_raw','judge','agency','docket','offense_date','id_year_proxy']),
 ('Disposed rows 2017–2024',d,['race_code','sex_code','class_raw','judge','agency','disposition_code','sentence_code']),
 ('Bail events 2017–2024',b,['race_code','sex_code','requested','set_amount','charge_list'])]:
    for c in cols:
        coverage.append({'population':name,'field':c,'rows':len(frame),'missing':int(frame[c].isna().sum()),'missing_pct':frame[c].isna().mean()*100})
coverage=pd.DataFrame(coverage)
table('field_blank_coverage',coverage,rows=30)
note(f'{len(quarantine):,} shifted or unattached rows across the charge extracts are quarantined, with coordinates/reasons only. Filing dates are absent. The dated analysis has {len(d):,} disposition rows and {len(b):,} bail events; the filing proxy-year cohort contains {len(f):,} charge rows.')

In [ ]:
outcome_map={'GPL':'Guilty plea','GJT':'Trial conviction','GBT':'Trial conviction','GJW':'Trial conviction',
 'CWF':'CWOF','PTP':'Recorded pretrial probation','NGBT':'Acquittal marker','NGJT':'Acquittal marker',
 'NGJW':'Acquittal marker','NGI':'Acquittal / insanity marker','NOP':'Nolle prosequi','NPDPH':'Nolle prosequi',
 'DIS':'Dismissal','OBJ':'Dismissal','NPC':'No probable cause','NBL':'No bill',
 'NR':'Civil not responsible','RESP':'Civil responsible','DEL':'Juvenile adjudication','ND':'Not delinquent',
 'TAC':'Procedural / transfer','BO':'Procedural / transfer','TYO':'Procedural / transfer',
 'RRDPH':'Procedural / review','FILE':'Filed: finding unclear','FWC':'Filed without change of plea'}
conviction_codes=['GPL','GJT','GBT','GJW']
trial_codes=['GJT','GBT','GJW','NGJT','NGBT','NGJW']
rank_class={'F':4,'M+':3,'M-':2,'CM':1}
for frame in [f_all,d_all,f,d]:
    frame['outcome']=frame.disposition_code.map(outcome_map).fillna('Missing / unmapped outcome')
    frame['conviction_marker']=frame.disposition_code.isin(conviction_codes)
    frame['class_rank']=frame.class_raw.map(rank_class).fillna(0)
sentence_map={'TERM':'Incarceration marker','SPLT':'Incarceration marker','LF':'Incarceration marker','LWOP':'Incarceration marker',
              'SS':'Suspended (probation unrecorded)','F':'Filed (not Fine)'}
for frame in [f_all,d_all,f,d]:
    frame['sentence_group']=frame.sentence_code.map(sentence_map)
    frame['sentence_group']=frame.sentence_group.where(frame.sentence_code.notna(),'Missing')
    frame['sentence_group']=frame.sentence_group.fillna('Unmapped sentence code')

def consensus(s):
    values=s.dropna().astype(str).unique()
    return 'Missing' if len(values)==0 else values[0] if len(values)==1 else 'Conflicting values'

def build_cases(frame):
    frame=frame.assign(class_missing=frame.class_raw.isna())
    g=frame.groupby('case_person',sort=False)
    c=g.agg(case_id=('case_id','first'),person_id=('person_id','first'),recorded_charge_rows=('case_id','size'),
      declared_charges=('declared_charge_count','first'),known_criminal=('known_criminal','max'),
      any_missing_class=('class_missing','max'),any_unmapped_class=('class_unmapped','max'),
      offense_date=('offense_date','min'),last_disposition=('disposition_date','max'),
      id_year_proxy=('id_year_proxy','first'))
    r=frame.sort_values(['case_person','class_rank','charge_code'],ascending=[True,False,True]).drop_duplicates('case_person').set_index('case_person')
    c['representative_code']=r.charge_code;c['highest_known_class']=r.class_raw.fillna('Missing')
    c['case_class']=c.highest_known_class
    no_felony=~c.highest_known_class.eq('F')
    c.loc[no_felony&c.any_missing_class,'case_class']='Incomplete: Missing class'
    c.loc[no_felony&c.any_unmapped_class,'case_class']='Unmapped class present'
    c.loc[no_felony&c.any_missing_class&c.any_unmapped_class,'case_class']='Missing + unmapped class'
    c['class_complete']=~(c.any_missing_class|c.any_unmapped_class)
    for column in ['race','sex','court','judge','case_status']:
        distinct=g[column].nunique(dropna=True)
        c[column]=g[column].first().fillna('Missing').where(distinct.le(1),'Conflicting values')
    closed=frame.case_status.notna()&frame.case_status.isin(['Closed','Closed YO','Closed Civil Ca','Completed'])
    c['closed_status']=closed.groupby(frame.case_person).all()
    c['invalid_disposition_date']=frame.disposition_date_quality.eq('Invalid / out of bounds').groupby(frame.case_person).any()
    return c
fc_all=build_cases(f_all)
fc=fc_all.loc[fc_all.id_year_proxy.between(2017,2024)].copy()
dc_all=build_cases(d_all)
cr=d_all.loc[d_all.known_criminal]
outcome_order=['Missing / unmapped outcome','Filed: finding unclear','Filed without change of plea','Procedural / review',
 'Procedural / transfer','No bill','No probable cause','Civil not responsible','Civil responsible','Dismissal','Nolle prosequi',
 'Not delinquent','Acquittal / insanity marker','Acquittal marker','Recorded pretrial probation','CWOF','Juvenile adjudication','Guilty plea','Trial conviction']
outcome_rank={x:i for i,x in enumerate(outcome_order)}
unclear={'Missing / unmapped outcome','Filed: finding unclear','Filed without change of plea','Procedural / review'}
def roll_outcome(s):
    values=set(s)
    if 'Trial conviction' in values:return 'Trial conviction'
    if values&unclear:return 'Missing / unresolved case outcome'
    return max(values,key=lambda x:outcome_rank[x])
dominant=cr.outcome.map(outcome_rank).groupby(cr.case_person).max().map({v:k for k,v in outcome_rank.items()})
unclear_case=cr.outcome.isin(unclear).groupby(cr.case_person).any()
dc_all['outcome']=dominant.mask(unclear_case&~dominant.eq('Trial conviction'),'Missing / unresolved case outcome')
dc_all['any_conviction']=cr.groupby('case_person').conviction_marker.max().reindex(dc_all.index,fill_value=False)
def roll_sentence(g):
    rows=g.loc[g.conviction_marker]
    if rows.empty:return 'Not eligible'
    values=set(rows.sentence_group)
    if 'Incarceration marker' in values:return 'Incarceration marker'
    if 'Missing' in values:return 'Missing / incomplete'
    if 'Unmapped sentence code' in values:return 'Unmapped sentence code'
    if 'Filed (not Fine)' in values:return 'Filed (not Fine)'
    return 'Suspended (probation unrecorded)'
sentence_ct=pd.crosstab(cr.loc[cr.conviction_marker].case_person,cr.loc[cr.conviction_marker].sentence_group)
sentence_case=pd.Series(np.select([
 sentence_ct.get('Incarceration marker',pd.Series(0,index=sentence_ct.index)).gt(0),
 sentence_ct.get('Missing',pd.Series(0,index=sentence_ct.index)).gt(0),
 sentence_ct.get('Unmapped sentence code',pd.Series(0,index=sentence_ct.index)).gt(0),
 sentence_ct.get('Filed (not Fine)',pd.Series(0,index=sentence_ct.index)).gt(0)],
 ['Incarceration marker','Missing / incomplete','Unmapped sentence code','Filed (not Fine)'],
 default='Suspended (probation unrecorded)'),index=sentence_ct.index)
dc_all['sentence_group']=sentence_case.reindex(dc_all.index,fill_value='Not eligible')
dc=dc_all.loc[dc_all.known_criminal&dc_all.closed_status&~dc_all.invalid_disposition_date&in_window(dc_all.last_disposition)].copy()
dc['year']=dc.last_disposition.dt.year.astype(int)
dc['offense_to_disposition_days']=(dc.last_disposition-dc.offense_date).dt.days
conv=dc.loc[dc.any_conviction].copy()
dc['outcome_unresolved']=dc.outcome.eq('Missing / unresolved case outcome')
conv['sentence_unresolved']=~conv.sentence_group.eq('Incarceration marker')
# Only incarceration vs suspended/filed/missing is directly recorded; probation/fine buckets are not complete.
table('case_population_flow',pd.Series({'full_filing_case_units':len(fc_all),'filing_ID_year_proxy_case_units':len(fc),
 'full_disposed_case_units':len(dc_all),'known_criminal_closed_case_units_latest_disposition_in_window':len(dc),
 'source_indicated_conviction_case_units':len(conv),'case_outcome_unresolved':int(dc.outcome_unresolved.sum()),
 'class_incomplete_disposed_cases':int((~dc.class_complete).sum())},name='cases'))
table('case_severity_completeness',dc.groupby(['case_class','class_complete']).size().to_frame('cases'))
table('outcome_code_description_pairs',d.groupby(['disposition_code','disposition_description'],dropna=False).size().to_frame('charge_rows'),rows=40)
table('sentence_code_description_pairs',d.groupby(['sentence_code','sentence_description'],dropna=False).size().to_frame('charge_rows'),rows=25)
table('disposition_vs_imposed_sentence_code',pd.crosstab(d.disposition_code.fillna('Missing'),d.sentence_code.fillna('Missing')),rows=40)
potential_mismatch=d.sentence_code.isin(['TERM','SPLT','LF','LWOP'])&~d.disposition_code.isin(conviction_codes+['DEL'])
table('potential_disposition_sentence_temporal_mismatch',d.loc[potential_mismatch].groupby(['disposition_code','sentence_code'],dropna=False).size().to_frame('charge_rows'),rows=35)
class_variants=f_all.groupby('charge_code').agg(nonmissing_class_variants=('class_raw','nunique'),
 source_description_variants=('charge_description','nunique'),charge_rows=('case_id','size'))
table('charge_codes_with_multiple_source_classes',class_variants.loc[class_variants.nonmissing_class_variants>1].sort_values('charge_rows',ascending=False),rows=35)
names=sorted(d_all.court.dropna().astype(str).unique())
prefix_pairs=[{'shorter_label':a,'longer_label':z} for a in names for z in names if len(a)<len(z) and z.startswith(a)]
table('possible_truncated_court_label_pairs',pd.DataFrame(prefix_pairs),rows=20)
note(f'{potential_mismatch.sum():,} disposition-window rows have a custodial sentence-type marker alongside a non-conviction disposition code. These are review flags, not proven errors: sentence fields may reflect an earlier event. Outcome coding follows the explicit disposition; old sentence fields never turn CWF/PTP/NR/DIS into convictions. '+
     f'{class_variants.nonmissing_class_variants.gt(1).sum():,} charge codes have multiple nonmissing source-class values; history or coding differences require review, not automatic reassignment.')
table('case_vs_docket_audit',pd.Series({'full_filing_case_defendant_units':len(fc_all),
 'full_filing_nonblank_docket_defendant_pairs':f_all.loc[f_all.docket.notna(),['docket','person_id']].drop_duplicates().shape[0],
 'full_filing_case_units_with_multiple_dockets':int((f_all.groupby('case_person').docket.nunique()>1).sum())},name='count'))

In [ ]:
b['bail_code_category']=text_category(b.bail_code)
bail_map={'PRBW':'Personal recognizance','PR':'Personal recognizance','BAIL':'Bail label (mode unspecified)',
          'IC':'In custody','ICBR':'Bail revoked / custody','WO':'Held without bail','WODN':'Dangerousness / custody'}
b['status_group']=b.bail_code.map(bail_map).fillna('Other recorded status')
b.loc[b.bail_code.isna(),'status_group']='Missing'
b['positive_amount']=b.set_amount.gt(0)
b['charge_tokens']=b.charge_list.map(lambda x:[] if pd.isna(x) else [t.strip() for t in str(x).split(',') if t.strip()])
b['parsed_code_count']=b.charge_tokens.map(len)
b['possible_truncated_code_list']=b.charge_list.map(lambda x:False if pd.isna(x) else len(str(x))>=45 and not str(x).endswith(','))
first=b.sort_values(['bail_date','bail_code_category','source_row']).drop_duplicates('case_id').copy()
pair=b.loc[b.requested.notna()&b.set_amount.notna()&b.requested.ge(0)&b.set_amount.ge(0)].copy()
pair['comparison']=np.select([pair.set_amount.lt(pair.requested),pair.set_amount.eq(pair.requested)],['Lower','Equal'],default='Higher')
identity=pd.concat([f_all[['case_id','person_id','case_person']],d_all[['case_id','person_id','case_person']]]).drop_duplicates()
safe_ids=identity.groupby('case_id').person_id.nunique().loc[lambda s:s==1].index
identity=identity.loc[identity.case_id.isin(safe_ids)].drop_duplicates('case_id')
first_link=first.merge(identity[['case_id','case_person']],on='case_id',how='left',validate='one_to_one')
first_link=first_link.merge(fc_all[['case_class','representative_code','known_criminal','recorded_charge_rows']],left_on='case_person',right_index=True,how='left',validate='many_to_one')
table('bail_charge_list_quality',pd.Series({'dated_events':len(b),'missing_code_list':int(b.charge_list.isna().sum()),
 'possible_truncated_code_list_heuristic':int(b.possible_truncated_code_list.sum()),
 'max_parsed_tokens':int(b.parsed_code_count.max())},name='events_or_count'))

In [ ]:
ANSWERS={};CONTRACTS=[]
def answer(number,status,text,limit,unit,denominator):
    ANSWERS[number]={'question':number,'status':status,'answer':text,'limitation_or_missing_input':limit,'unit':unit,'denominator':denominator}
    display(Markdown(f'**Answerability: {status}.** {text}\n\n**Limit / missing input:** {limit}'))
def contract(number,visual,unit,period,num,den,missing,extra='None'):
    CONTRACTS.append({'question':number,'uniform_visual':visual,'unit':unit,'period':period,'numerator':num,
      'denominator':den,'missing_handling':missing,'additional_input_needed':extra})
outcome_missing=dc.outcome_unresolved.mean()
sentence_missing=conv.sentence_group.eq('Missing / incomplete').mean()

## Q1. How many cases were referred to the prosecutor each year, and how has that changed over the study period?

**Scope: Volume and scope.** No referral intake records or filing dates are supplied. The only annual filing-extract companion uses an unverified case-ID prefix. It cannot be called referred cases or a verified filing trend.

In [ ]:
proxy=fc.groupby('id_year_proxy').size().to_frame('case_defendant_units')
table('Q01_unverified_ID_year_companion',proxy)
fig,ax=plt.subplots(figsize=(10,4.5));proxy.plot(ax=ax,marker='o',legend=False)
ax.set(title='Companion only: case counts by unverified ID-year prefix',xlabel='Case-ID-year proxy — not filing/referral date',ylabel='Recorded case-defendant units')
finish(fig,'Q01_ID_year_proxy','Only matching case-ID prefixes 2017–2024. IDs without a year proxy remain in full-file Missing audit; no verified annual referrals or filings.')
answer(1,'Not answerable','Referral counts cannot be established. An unverified case-ID-year filing-extract companion is supplied separately.',
 'No referral population, screening outcomes, or filing-date field. A case-ID prefix has not been validated as filing year.',
 'Case-defendant proxy for the companion','No complete referred-case denominator')
contract(1,'Annual referrals/filings only after dates are supplied','Case','Referral/filing year unavailable','Documented referred/initiated cases','All referrals/filings','Missing dates stay unavailable','Referral records and verified filing dates')

## Q2. What is the per-capita case rate (cases per 10,000 residents), so counties of different sizes can be compared?

**Scope: Volume and scope.** No population figures occur in the three local files. No outside population series is loaded or inferred.

In [ ]:
answer(2,'Not answerable','No cases-per-10,000-residents rate is calculated.',
 'Annual population is absent; annual filing counts are also not verified because filing dates are missing.',
 'Cases per residents','Annual resident population — absent')
contract(2,'Annual rate line after inputs exist','Case / residents','2017–2024','Verified annual filed cases','Annual population','No guessed population/date','External annual population plus filing dates; not used here')

## Q3. What share of cases are felonies vs. misdemeanors, and what are the 5 most common offense categories?

**Scope: Volume and scope.** The source has Missing and nonstandard charge categories; no external crime dictionary is used. One case’s highest known class is not treated as definite if another charge has missing/unmapped class, unless a known felony already establishes the broad felony marker. Top-five representative codes are a proxy for offense categories.

In [ ]:
classes=dc.case_class.value_counts().to_frame('case_units')
class_unclear=~dc.case_class.isin(['F','M+','M-'])
classes['percent_observed_cases']=classes.case_units/len(dc)*100 if class_unclear.mean()<=.2 else np.nan
table('Q03_case_class_with_missing_and_unmapped',classes)
top=dc.representative_code.value_counts().head(5)
table('Q03_top_five_representative_codes',top.rename(index=labels).to_frame('case_units'))
fig,ax=barh(classes.case_units,'Observed disposed cases: severity evidence and incomplete class','Case-defendant units')
finish(fig,'Q03_case_class','Known-criminal closed disposition-window cohort, not a filing cohort. Missing/unmapped-only cases are not declared civil or excluded without disclosure.')
fig,ax=barh(top.rename(index=labels),'Companion: five representative allegation codes','Case-defendant units')
finish(fig,'Q03_representative_codes','Representative code is a highest-known-source-class proxy with deterministic ties, not validated statutory seriousness or a broad offense-category crosswalk.')
answer(3,'Partial',f'Case-level class counts and five representative codes are available in the observed disposed cohort; {class_unclear.mean():.1%} cannot be assigned a definite felony/misdemeanor class under the conservative rule.',
 'No verified filing cohort, broad offense categories, or within-class seriousness. Missing/unmapped source classes remain explicit; dependent shares are withheld above 20%.',
 'Case-defendant proxy','Observed closed cases with ≥1 recognized criminal charge')
contract(3,'Source-class and incompleteness bar plus top-five code bars','Case','Disposition year, not filing year','Cases by definite source class or representative code','Selected known-criminal observed cases','Missing and unmapped separate; >20% blocks shares','Verified filing dates, class/codebook and offense-category crosswalk')

## Q4. What percentage of referred cases does the prosecutor decline to charge?

**Scope: Charging decisions.** Nolle prosequi, dismissal, no bill, and pretrial probation are recorded later outcomes; none supplies a complete referral-screening denominator.

In [ ]:
answer(4,'Not answerable','Pre-filing declination rate cannot be calculated.',
 'No all-referrals table or documented decision to decline before filing. Do not substitute post-filing dismissals/nolle prosequi.',
 'Referred case','All referred cases — absent')
contract(4,'Declination/charged/unknown chart after intake data','Referred case','Referral year','Pre-filing declines','All referrals','Unknown decision retained','Referral intake and screening outcomes')

## Q5. How often is the filed charge different from the arrest/referral charge (upgraded, downgraded, unchanged)?

**Scope: Charging decisions.** Cape has neither original arrest/referral charges nor a stable exported charge sequence/version ID. A filing-versus-disposition code overlap is not an original-to-filed change history.

In [ ]:
answer(5,'Not answerable','No upgrade/downgrade/unchanged rate is reported.',
 'Original charges, amendment versions, charge sequence identity, and validated seriousness are absent. Matching on code would select unchanged codes by construction.',
 'Charge','Charges with both original and filed versions — absent')
contract(5,'Three-category charge-change chart after versioned data','Charge','Referral to filing','Upgraded/downgraded/unchanged','All properly version-linked charges','Unknown transition retained','Original charges, stable IDs, amendment history and seriousness')

## Q6. What share of cases are diverted to alternative programs instead of prosecuted?

**Scope: Charging decisions.** PTP is explicitly described as Pretrial Probation in the source, whereas CWF is Continued without Finding. The observable PTP marker is not automatically enrollment in an alternative program instead of prosecution.

In [ ]:
ptp_cases=set(d_all.loc[d_all.disposition_code.eq('PTP')].case_person)
recorded_ptp=dc.index.isin(ptp_cases)
ptp_table=pd.Series({'observed_closed_case_units':len(dc),'cases_with_any_recorded_PTP':int(recorded_ptp.sum()),
 'cases_with_dominant_PTP_outcome':int(dc.outcome.eq('Recorded pretrial probation').sum()),
 'cases_with_dominant_CWOF_outcome_separate':int(dc.outcome.eq('CWOF').sum())},name='case_units')
table('Q06_PTP_CWOF_evidence',ptp_table)
fig,ax=barh(dc.outcome.loc[dc.outcome.isin(['Recorded pretrial probation','CWOF'])].value_counts(),
 'Recorded PTP and CWOF are separate case outcomes','Case-defendant units')
finish(fig,'Q06_PTP_CWOF','Dominant observed case outcomes under the stated hierarchy. PTP is not verified program enrollment; CWOF is not reclassified as diversion.')
answer(6,'Partial',f'{recorded_ptp.sum():,} observed closed known-criminal case units have at least one PTP marker. CWOF is reported separately.',
 'No program eligibility, referral, enrollment, completion, or complete all-charged-case denominator. The literal alternative-program diversion rate is unavailable.',
 'Observed case-defendant proxy','Observed closed known-criminal cases for the marker companion')
contract(6,'Separate PTP/CWOF recorded-marker counts','Case','Disposition year','Documented PTP markers; CWOF separate','Observed disposed cases, not program eligibility','Do not infer zero/participation from missing','Program intake/enrollment and eligibility for literal question')

## Q7. Of charged cases, what percentage end in dismissal, plea, trial conviction, or acquittal?

**Scope: Case outcomes.** Use a case-level hierarchy on the observed closed known-criminal cohort. This is not the eventual outcome distribution of all charged cases. Missing/unmapped severity limits the criminal selection; civil responsibility and juvenile adjudication remain separate.

In [ ]:
outcomes=dc.outcome.value_counts().to_frame('case_units')
outcomes['percent_of_observed_cases']=outcomes.case_units/len(dc)*100 if outcome_missing<=.2 else np.nan
table('Q07_case_outcomes',outcomes,rows=30)
by_year=pd.crosstab(dc.year,dc.outcome);table('Q07_case_outcomes_by_year',by_year)
fig,ax=barh(outcomes.case_units,'Observed case outcomes, with Missing/unresolved explicit','Case-defendant units')
finish(fig,'Q07_case_outcomes','Closed known-criminal case proxy; latest recorded disposition in 2017–2024. Hierarchy requires team agreement; unresolved charges are not silently dropped.')
fig,ax=stacked(by_year,'Observed case outcomes by latest disposition year');ax.set_ylabel('Case-defendant units')
finish(fig,'Q07_case_outcome_year','Counts, not all-charged-case eventual outcome rates. Civil and juvenile results are not adult conviction/acquittal.')
answer(7,'Partial',f'{len(dc):,} observed case units have a provisional outcome; {outcome_missing:.1%} are unresolved. '+
 ('Provisional shares retain unresolved cases.' if outcome_missing<=.2 else 'Percentage claims are withheld because unresolved coverage exceeds 20%.'),
 'No complete eventual follow-up or verified filing cohort; case closure, criminal scope and highest-outcome rules remain qualified. PTP is a source marker, not program enrollment.',
 'Case-defendant proxy','Observed closed known-criminal cases with latest valid disposition in 2017–2024')
contract(7,'Outcome counts/shares with missing and separate CWOF/PTP','Case','Latest disposition year','Highest agreed recorded outcome','Observed disposed criminal cases','Unknown retained; >20% blocks rates','Agreed case/docket, severity and outcome crosswalk')

## Q8. What is the plea rate, and how often do pleas involve a reduced charge?

**Scope: Case outcomes.** GPL explicitly means Guilty Plea in this export. Count cases with at least one GPL; do not count CWOF/PTP as GPL. Reduced-charge information is absent.

In [ ]:
gpl=set(d_all.loc[d_all.disposition_code.eq('GPL')].case_person)
dc['any_GPL']=dc.index.isin(gpl)
pleas=dc.groupby('year').any_GPL.agg(cases='size',GPL_cases='sum');pleas['GPL_marker_pct']=pleas.GPL_cases/pleas.cases*100
table('Q08_GPL_case_markers',pleas)
fig,ax=plt.subplots(figsize=(10,4.5));ax.plot(pleas.index,pleas.GPL_marker_pct,marker='o')
ax.set(title='Observed cases with at least one Guilty Plea code',xlabel='Latest disposition year',ylabel='Percent of observed known-criminal cases')
finish(fig,'Q08_GPL_markers','Source GPL marker, one case once. Not an all-charged-case plea probability; PTP/CWF are separate and charge reduction is unknown.')
answer(8,'Partial',f'{dc.any_GPL.sum():,} of {len(dc):,} observed case units ({dc.any_GPL.mean():.1%}) have a GPL marker.',
 'No original charges, plea agreements or version history, so reduced-charge frequency is unavailable. No complete all-charged-case denominator.',
 'Case-defendant proxy','Observed closed known-criminal cases')
contract(8,'Annual GPL case marker share','Case','Disposition year','Cases with ≥1 GPL','Observed disposed cases','CWF/PTP excluded from GPL','Original charge/plea agreement and full filing cohort for literal question')

## Q9. What share of cases go to trial, and what is the conviction rate at trial?

**Scope: Case outcomes.** Trial route and guilty/not-guilty markers are explicit for jury, bench, and jury-waived codes. Compare within that observed marker population; do not infer trial from generic procedural outcomes or convert civil NR to acquittal.

In [ ]:
trial_rows=d_all.loc[d_all.known_criminal&d_all.disposition_code.isin(trial_codes)]
trial_case=trial_rows.groupby('case_person').disposition_code.agg(lambda s:'At least one trial conviction marker' if s.isin(['GJT','GBT','GJW']).any() else 'Only not-guilty trial markers')
tc=dc.loc[dc.index.isin(trial_case.index)].copy();tc['trial_result']=trial_case.reindex(tc.index)
trial_table=tc.trial_result.value_counts().to_frame('cases');trial_table['percent_explicit_trial_cases']=trial_table.cases/len(tc)*100
table('Q09_explicit_trial_case_results',trial_table)
fig,ax=barh(trial_table.cases,'Results among explicitly trial-coded observed cases','Case-defendant units')
finish(fig,'Q09_explicit_trial_results','A case with mixed trial verdicts counts as having a conviction marker. No trial-event history or all-charged-case denominator is supplied.')
trial_rate=trial_table.loc['At least one trial conviction marker','percent_explicit_trial_cases'] if 'At least one trial conviction marker' in trial_table.index else 0
answer(9,'Partial',f'{len(tc):,} observed case units ({len(tc)/len(dc):.1%}) have an explicit trial code. Within these, {trial_rate:.1f}% have at least one trial conviction marker.',
 'This is the explicit-code resolved-case subset, not the share of all charged cases going to trial. Mixed verdicts, extraction selection and incomplete histories limit interpretation.',
 'Case-defendant proxy','Explicit trial-coded observed closed cases for the result share')
contract(9,'Trial-marker result bars with denominator','Case','Disposition year','Any explicit trial guilty vs not-guilty markers','Explicit trial-coded observed cases','NR/RESP never treated as criminal verdicts','Complete trial and all-charged-case history for full question')

## Q10. What share of convictions result in incarceration vs. probation vs. fines/other?

**Scope: Sentencing.** The type codes do not furnish a complete incarceration/probation/fines classification. F is Filed, SS is Suspended, and blanks do not establish noncustody. Check missingness within source-indicated conviction cases before reporting a rate.

In [ ]:
sentences=conv.sentence_group.value_counts().to_frame('conviction_marker_cases')
table('Q10_sentence_evidence_and_missing',sentences)
fig,ax=barh(sentences.iloc[:,0],'Sentence evidence and Missing among conviction-marker cases','Case-defendant units')
finish(fig,'Q10_sentence_coverage','No noncustodial inference from Missing. Suspended is not verified probation; Filed is not Fine. Source markers are not actual prison admissions.')
answer(10,'Insufficient data' if sentence_missing>.2 else 'Partial',
 f'{sentence_missing:.1%} of {len(conv):,} conviction-marker case units have Missing/incomplete type. '+
 ('This fails the 20% rule; requested sentence-type percentage claims are withheld.' if sentence_missing>.2 else 'Known source type counts are shown, but the requested three-bucket mapping remains incomplete.'),
 'No complete probation/fine fields; sparse imposed types, filed/suspended labels and unmapped age-limited codes prevent a full comparable sentence taxonomy.',
 'Source-indicated conviction case','Observed cases with explicit GPL/GJT/GBT/GJW conviction code')
contract(10,'Sentence evidence and Missing counts; no unsupported shares','Convicted case','Disposition year','Recorded case sentence type','Source-indicated conviction cases','>20% missing blocks rates; F≠Fine','Complete sentence types and probation/fine data')

## Q11. What is the median sentence length by offense category?

**Scope: Sentencing.** Total case sentence length is not provided. A limited companion uses the longest uniquely labeled Actual MAX jail term per finite custodial conviction case. Life/age-limited sentences, concurrent/consecutive structure, and ambiguous duplicate unit headers are not guessed.

In [ ]:
cust=d_all.loc[d_all.conviction_marker&d_all.sentence_group.eq('Incarceration marker')].copy()
cust['term_value']=pd.to_numeric(cust['Imposed Actual MAX Term'],errors='coerce')
cust['factor']=cust['Imposed Actual MAX Term Unit'].map({'Day':1,'Week':7,'Month':365.25/12,'Year':365.25,'Hour':1/24})
cust['actual_max_days']=(cust.term_value*cust.factor).where(cust.term_value.ge(0))
life_ids=set(cust.loc[cust.sentence_code.isin(['LF','LWOP'])].case_person)
finite=conv.loc[conv.sentence_group.eq('Incarceration marker')&~conv.index.isin(life_ids)].copy()
finite['recorded_days']=cust.groupby('case_person').actual_max_days.max().reindex(finite.index)
summary=finite.groupby('case_class').agg(cases=('case_id','size'),usable=('recorded_days','count'),median_available=('recorded_days','median'))
summary['missing_pct']=(1-summary.usable/summary.cases)*100
summary['reportable_median']=summary.median_available.where((summary.missing_pct<=20)&(summary.usable>=MIN_CELL))
table('Q11_recorded_actual_term_companion',summary)
fig,ax=barh(summary.cases-summary.usable,'Missing finite actual-term evidence','Case-defendant units')
finish(fig,'Q11_duration_coverage','Finite custodial marker cases only. Longest observed Actual MAX is not total case sentence or time served; M/Y conversion is an analytical convention.')
report=summary.reportable_median.dropna()
if len(report):
    fig,ax=barh(report,'Reportable companion medians of recorded actual term','Converted days')
    finish(fig,'Q11_recorded_term_medians','Group requires ≥30 usable terms and ≤20% missing. No complete broad offense-category or total sentence calculation.')
answer(11,'Partial',f'{len(finite):,} finite custodial conviction-marker case units are assessed; {finite.recorded_days.isna().mean():.1%} lack a usable recorded Actual MAX term. Reportable source-class companion medians are shown where thresholds pass.',
 'No complete case sentence, concurrent/consecutive structure, validated offense-category map, or unambiguous Imposed MIN/MAX unit assignment. Life is not zero days.',
 'Finite custodial case proxy','Observed finite explicit custodial conviction cases')
contract(11,'Finite term coverage plus reportable median','Custodial case','Disposition year','Longest recorded Actual MAX term','Finite custodial cases, audited coverage','Keep Missing/unmapped/nonfinite separate','Full sentence structure and category map')

## Q12. What share of defendants are detained pretrial, and how often is cash bail set?

**Scope: Pretrial and timing.** Bail supplies case IDs, recorded status, amounts, and dates, but no actual custody duration, posting ledger or person ID. Its BAIL code does not establish cash-only/surety. County-wide defendant detention and cash-bail rates cannot be reconstructed.

In [ ]:
observed=first.status_group.value_counts().to_frame('observed_case_ids')
table('Q12_observed_first_bail_statuses',observed)
linked_closed=first_link.loc[first_link.case_person.isin(dc.index)]
table('Q12_dated_extract_overlap',pd.Series({'closed_disposition_case_units':len(dc),
 'with_observed_bail_event_in_window':len(linked_closed),
 'without_observed_bail_event_in_window':len(dc)-len(linked_closed)},name='case_units'))
fig,ax=barh(observed.iloc[:,0],'First observed bail/status profile, not all defendants','Observed case IDs')
finish(fig,'Q12_observed_bail_statuses','Dated first event per case, not guaranteed arraignment. Missing/Unknown retained; absent record is not release, and In Custody is not inability to pay.')
answer(12,'Partial',f'{len(first):,} case IDs have a first observed dated event; their recorded status profile is tabulated. {len(linked_closed):,} link to the dated closed known-criminal disposition cohort.',
 'No complete defendant arraignment/custody/payment history, cash-only/surety mode, or verified filing-cohort coverage. The overlap is not coverage of all charged defendants.',
 'Observed case ID, not unique defendant','Only cases with a recorded dated bail event for the companion')
contract(12,'Observed status and linkage counts, not county-wide detention rate','Case proxy','First valid event in 2017–2024','Recorded status labels','Observed first-event cases','Missing record ≠ released; no mode imputation','Complete defendant bail/custody/payment history and filing cohort')

## Q13. What is the median time from filing to disposition, and does it differ by case type?

**Scope: Pretrial and timing.** Neither charge file has a filing/initiation date. An offense-to-disposition interval is a different measure and is not substituted for processing time. No date is invented from a case-ID year or month.

In [ ]:
answer(13,'Not answerable','Median filing-to-disposition time cannot be calculated.',
 'Filing/initiation date is absent from both charge extracts. Offense date and case-ID year are not filing dates. Offense-to-disposition is shown only as a separate Q19 companion.',
 'Case','Cases with both verified filing and disposition dates — unavailable')
contract(13,'Median/IQR processing-time plot only after filing dates exist','Case','2017–2024','Disposition minus verified filing date','Valid resolved-case intervals','Missing filing dates are not imputed','Actual filing/initiation date and case-type definition')

## Q14. Do declination, dismissal, and incarceration rates differ by defendant race/ethnicity?

**Scope: Equity.** Case-level dismissal/nolle markers can be compared by recorded race within the selected disposed cohort, with Missing/Unknown/unmapped groups retained. Separate ethnicity and declinations are absent. Incarceration rates are suppressed where sentence missingness exceeds 20%.

In [ ]:
def equity(group):
    rows=[]
    for label,g in dc.groupby(group):
        v=conv.loc[conv.index.intersection(g.index)]
        missing_type=v.sentence_group.eq('Missing / incomplete').mean() if len(v) else np.nan
        rows.append({'group':label,'disposed_cases':len(g),'outcome_unresolved_pct':g.outcome_unresolved.mean()*100,
          'dismissal_nolle_pct':g.outcome.isin(['Dismissal','Nolle prosequi']).mean()*100 if len(g)>=MIN_CELL and g.outcome_unresolved.mean()<=.2 else np.nan,
          'conviction_marker_cases':len(v),'sentence_missing_pct':missing_type*100,
          'incarceration_marker_pct':v.sentence_group.eq('Incarceration marker').mean()*100 if len(v)>=MIN_CELL and missing_type<=.2 else np.nan})
    result=pd.DataFrame(rows).set_index('group');ref='White' if group=='race' else 'Male'
    for metric in ['dismissal_nolle_pct','incarceration_marker_pct']:
        baseline=result.loc[ref,metric] if ref in result.index else np.nan
        result[metric+'_ratio_vs_'+ref]=result[metric]/baseline if pd.notna(baseline) and baseline>0 else np.nan
    return result
race_eq=equity('race');table('Q14_race_outcome_markers',race_eq,rows=30)
plot=race_eq[['dismissal_nolle_pct','incarceration_marker_pct']].dropna(how='all')
fig,ax=plt.subplots(figsize=(11,5));plot.rename(columns={'dismissal_nolle_pct':'Dismissal/nolle','incarceration_marker_pct':'Incarceration marker'}).plot.bar(ax=ax)
ax.set(title='Race: reportable recorded case outcome markers',ylabel='Percent within stated eligible population',xlabel='');ax.tick_params(axis='x',rotation=35)
finish(fig,'Q14_race_outcomes','Dismissal denominator: observed disposed known-criminal cases. Incarceration denominator: conviction-marker cases. n<30/>20% missing suppresses rate; Missing stays a group.')
answer(14,'Partial','Reportable dismissal/nolle comparisons and White reference ratios are supplied. Incarceration shares are withheld for groups failing sentence coverage; declination is unavailable.',
 'No referral screening, separate ethnicity, population exposure, prior record or case facts. Source-class incompleteness selects the cohort; these are descriptive associations.',
 'Case-defendant proxy','Disposed cases for dismissal; conviction-marker cases for sentence markers')
contract(14,'Outcome-marker panels with White ratios and suppression','Case','Disposition year','Dismissal/nolle and observed incarceration markers','Eligible population separately stated','Missing/Unknown/unmapped retained; small and incomplete cells suppressed','Referral/ethnicity/prior-record and complete sentence data')

## Q15. Do those outcomes differ by sex and age group?

**Scope: Equity.** Use recorded sex, not inferred gender identity. No date of birth or age exists; age-limited sentence labels cannot reconstruct defendant age.

In [ ]:
sex_eq=equity('sex');table('Q15_recorded_sex_outcomes',sex_eq,rows=30)
plot=sex_eq[['dismissal_nolle_pct','incarceration_marker_pct']].dropna(how='all')
fig,ax=plt.subplots(figsize=(10,5));plot.rename(columns={'dismissal_nolle_pct':'Dismissal/nolle','incarceration_marker_pct':'Incarceration marker'}).plot.bar(ax=ax)
ax.set(title='Recorded sex: reportable case outcome markers',ylabel='Percent within stated eligible population',xlabel='');ax.tick_params(axis='x',rotation=25)
finish(fig,'Q15_sex_outcomes','Male reference ratios in the table. Missing/Unknown retained; no age-group inference and no causal or population-adjusted estimate.')
answer(15,'Partial','Recorded-sex comparisons with Male reference ratios are available for reportable cells; age-group results are unavailable.',
 'No age/date of birth, declination, complete sentence history, or complete gender identity measure. Q14 coverage and cohort-selection limitations apply.',
 'Case-defendant proxy','Same distinct denominators as Q14, by recorded sex')
contract(15,'Same outcome panels by recorded sex','Case','Disposition year','As Q14','As Q14','Missing/Unknown retained; suppress weak cells','Age/date of birth and complete sentence/referral data')

## Q16. Do racial disparities remain when you compare within the same offense category and prior-record level?

**Scope: Equity.** Earlier rows in a limited county extract do not provide a complete prior-record measure or the team’s uniform lookback. No prior history is reconstructed from case numbers.

In [ ]:
answer(16,'Not answerable','The requested offense-by-prior-record controlled comparison cannot be performed.',
 'No complete prior record, uniform lookback, or validated offense-category/seriousness crosswalk. Same-class comparisons alone do not answer this question.',
 'Case','Offense-category × prior-record cells — absent')
contract(16,'Controlled within-category/prior-record rates after inputs exist','Case','Common lookback','Recorded group outcomes','Comparable offense × prior-record cells','Suppress <30; retain unknown controls','Prior record with common lookback and offense-category map')

## Q17. Did any of the above metrics shift noticeably around a policy change, new DA term, or 2020?

**Scope: Trends and context.** Actual disposition-year changes can be described. No verified filing trend, policy dates, DA-term dates, or counterfactual is supplied. The year 2020 is a question-specified reference, not an estimated intervention.

In [ ]:
trend=dc.groupby('year').agg(observed_case_units=('case_id','size'))
trend['change_yoy_pct']=trend.observed_case_units.pct_change()*100
trend['index_2017_100']=trend.observed_case_units/trend.loc[2017,'observed_case_units']*100
table('Q17_disposition_year_trends',trend)
fig,ax=plt.subplots(figsize=(10,4.5));ax.plot(trend.index,trend.index_2017_100,marker='o');ax.axvline(2020,color=COLORS[3],ls='--')
ax.set(title='Observed closed-case dispositions around 2020',xlabel='Latest disposition year',ylabel='Index: 2017 = 100')
finish(fig,'Q17_disposition_trends','Disposition counts, not filings or crimes. The reference year does not establish a policy, DA-term, COVID, or other causal effect.')
change=(trend.loc[2020,'observed_case_units']/trend.loc[2019,'observed_case_units']-1)*100
answer(17,'Partial',f'Observed closed known-criminal disposition case counts changed {change:.1f}% from 2019 to 2020. Descriptive disposition-year trends are shown.',
 'No verified filing dates, policy/DA-term timeline or counterfactual. Differences can reflect backlog, case mix, extraction and recording changes.',
 'Observed case-defendant proxy','Dated closed known-criminal disposition cases each year')
contract(17,'Indexed disposition trend with explicit period label','Case','Disposition year','Observed annual disposed cases','2017 baseline','Keep classification/coverage limits visible','Filing dates, policy timeline and causal design for attribution')

## Q18. How does this county rank against the state average and the other counties on the team’s core metrics (rates from Q4, Q7, Q10, Q14)?

**Scope: Trends and context.** This is a Cape-only assessment. No Berkshire, other-county, statewide, or external numerical results are imported into these analytical notebooks.

In [ ]:
answer(18,'Not answerable','No state average or county ranking is calculated.',
 'Comparable other-county/statewide data and harmonized units/outcomes are absent; Cape lacks filing dates and complete sentencing fields.',
 'County','Comparable county/state eligible populations — absent')
contract(18,'County comparison only after harmonization','County','Common period','Agreed comparable outcomes','Common case and denominator contract','Do not rank incomparable or unavailable counties','Other-county/statewide records and common definitions')

## Q19. How does the number of recorded charges per case relate to the time from offense to disposition?

**Scope: Case complexity.** Aggregate the full filing extract to a case before linking to dated observed dispositions. The source has no ordinal count ID, so recorded rows are retained as potentially separate counts. This is an observed resolved-case association, not a verified filing-cohort or causal analysis.

In [ ]:
linked=fc_all[['recorded_charge_rows','declared_charges']].merge(dc[['offense_to_disposition_days']],left_index=True,right_index=True,how='inner',validate='one_to_one')
usable=linked.loc[linked.offense_to_disposition_days.ge(0)].copy()
usable['burden']=pd.cut(usable.recorded_charge_rows,[0,1,2,4,np.inf],labels=['1','2','3–4','5+'])
summary=usable.groupby('burden',observed=True).offense_to_disposition_days.agg(cases='size',median='median',p25=lambda s:s.quantile(.25),p75=lambda s:s.quantile(.75))
table('Q19_charge_complexity_and_offense_interval',summary)
table('Q19_linkage_and_interval_exclusions',pd.Series({'closed_disposition_cases':len(dc),'linked_to_filing_extract':len(linked),
 'usable_nonnegative_intervals':len(usable),'missing_or_negative_intervals':len(linked)-len(usable)},name='cases'))
fig,ax=plt.subplots(figsize=(10,5));ax.errorbar(range(len(summary)),summary['median'],yerr=[summary['median']-summary.p25,summary.p75-summary['median']],fmt='o',capsize=5)
ax.set_xticks(range(len(summary)),[f'{label} (n={n})' for label,n in zip(summary.index,summary.cases)])
ax.set(title='Recorded charge burden and offense-to-disposition interval',xlabel='Recorded filing-extract charge rows per case',ylabel='Median days and IQR')
finish(fig,'Q19_complexity_interval','One case per observation; identical rows retained. Not filing-to-disposition time. Unmatched/invalid and still-open cases create selection.')
answer(19,'Partial',f'{len(usable):,} linked case units have usable offense-to-disposition intervals; medians/IQRs by recorded charge burden are supplied.',
 'No verified filing cohort, stable charge ordinal history, complete case facts or unresolved-case follow-up. It is not a causal effect or forecast.',
 'Linked case-defendant proxy','Observed dated closed known-criminal cases linked to filing extract')
contract(19,'Median/IQR by charge burden','Case','Disposition in 2017–2024; full filing-extract link','Offense-to-latest-disposition interval','Usable observed linked cases','Report unmatched/missing/negative; retain possible true repeat counts','Complete charge history and follow-up for full population')

## Q20. Do dismissal, conviction, and diversion rates differ across courts when comparing cases of the same offense severity?

**Scope: Court and agency patterns.** Compare cases only where broad source class is definite and one court is recorded. Missing/unmapped severity cannot be silently filled. Courts with truncated or conflicting labels remain separate; PTP is a recorded marker, not proven diversion-program enrollment.

In [ ]:
rows=[]
eligible=dc.loc[dc.case_class.isin(['F','M+','M-'])&~dc.court.isin(['Missing','Conflicting values'])]
for (severity,court),g in eligible.groupby(['case_class','court']):
    good=len(g)>=MIN_CELL and g.outcome_unresolved.mean()<=.2
    rows.append({'source_class':severity,'court':court,'cases':len(g),'outcome_unresolved_pct':g.outcome_unresolved.mean()*100,
      'dismissal_nolle_pct':g.outcome.isin(['Dismissal','Nolle prosequi']).mean()*100 if good else np.nan,
      'conviction_marker_pct':g.any_conviction.mean()*100 if good else np.nan,
      'PTP_outcome_pct':g.outcome.eq('Recorded pretrial probation').mean()*100 if good else np.nan})
comparison=pd.DataFrame(rows);table('Q20_court_same_source_class',comparison,rows=40)
for metric,title in [('dismissal_nolle_pct','Dismissal/nolle'),('conviction_marker_pct','Conviction markers'),('PTP_outcome_pct','Recorded PTP outcomes')]:
    matrix=comparison.pivot(index='court',columns='source_class',values=metric)
    fig,ax=heatmap(matrix,f'{title}: courts within definite source class',percent=True,annotate=True)
    finish(fig,'Q20_'+metric,'Observed case-level cells; n<30 or >20% unresolved suppressed. Missing/unmapped severity and conflicting courts excluded and audited; not complete seriousness adjustment.')
answer(20,'Partial','Same-definite-source-class dismissal/nolle, conviction-marker and recorded-PTP comparisons are supplied for reportable court cells.',
 'No validated within-class seriousness, program participation or prior record. Court labels/assignment and missing severity limit comparability; PTP is not a full diversion measure.',
 'Observed case-defendant proxy','Court × definite source-class case cells')
contract(20,'Court × source-class heatmaps with coverage','Case','Disposition year','Dismissal/nolle, conviction and PTP markers','Eligible court × class cells','No class imputation; n≥30 and unresolved≤20%','Validated seriousness, court aliases, program and prior-record data')

## Q21. Which agencies account for the largest share of filed charges, and how does their offense mix differ?

**Scope: Court and agency patterns.** Recorded agency and charge mix are available, but no filing-date field supports the common 2017–2024 filing filter. The whole filing-extract composition is an explicitly labeled companion; Missing/unmapped severity stays visible.

In [ ]:
agency=f_all.agency.fillna('Missing').value_counts().to_frame('charge_rows')
agency['share_full_extract_pct']=agency.charge_rows/len(f_all)*100
table('Q21_full_filing_extract_agency_companion',agency,rows=30)
mix=pd.crosstab(f_all.agency.fillna('Missing'),f_all.class_plot)
table('Q21_agency_source_class_mix',mix,rows=30)
table('Q21_agency_charge_code_counts',pd.crosstab(f_all.agency.fillna('Missing'),f_all.charge_code),rows=12)
fig,ax=barh(agency.charge_rows.head(10),'Companion: agencies in the full filing extract')
finish(fig,'Q21_full_extract_agencies','No verified filing-date window. Missing agency remains in the denominator; rows describe recorded charge workload, not all referrals or agency performance.')
fig,ax=stacked(mix.reindex(agency.head(10).index),'Full-extract agencies: source class including Missing',percent=True)
finish(fig,'Q21_agency_class_mix','Full filing-extract charge rows. Cannot be directly pooled with verified 2017–2024 filing-year populations in other counties.')
answer(21,'Partial',f'In the full clean filing extract, {agency.index[0]} supplies {agency.iloc[0].share_full_extract_pct:.1f}% of charge rows. Class/code mix is tabulated.',
 'No verified filing dates, so a common 2017–2024 filed-charge denominator cannot be applied. No broad offense-category crosswalk or complete referrals.',
 'Charge row','All clean full filing-extract rows, not a verified filing-year cohort')
contract(21,'Agency volume and class composition after a common date rule','Charge','Full extract only until dates supplied','Agency charge rows by source class/code','All full-extract rows; agency rows for mix','Missing agency/class and unmapped class explicit','Verified filing dates for common 2017–2024 comparison')

## Q22. When both amounts are recorded, how often is bail set lower than, equal to, or higher than the amount requested?

**Scope: Bail decisions.** The literal question explicitly conditions on both amounts being recorded. It can be answered from dated bail events, with the limited pair coverage and repeated-event sensitivity disclosed.

In [ ]:
paired=pair.comparison.value_counts().reindex(['Lower','Equal','Higher'],fill_value=0).to_frame('events')
paired['percent_of_paired_events']=paired.events/len(pair)*100
table('Q22_paired_bail_comparison',paired)
one=pair.sort_values(['bail_date','source_row']).drop_duplicates('case_id').comparison.value_counts().reindex(['Lower','Equal','Higher'],fill_value=0).to_frame('case_ids')
one['percent_first_paired_case']=one.case_ids/one.case_ids.sum()*100
table('Q22_first_paired_event_per_case_sensitivity',one)
fig,ax=plt.subplots(figsize=(9,5));ax.bar(paired.index,paired.percent_of_paired_events,color=[COLORS[1],COLORS[0],COLORS[3]])
for i,value in enumerate(paired.percent_of_paired_events):ax.text(i,value+1,f'{value:.1f}%\nn={paired.iloc[i].events:,.0f}',ha='center')
ax.set(title='Bail set versus requested: paired recorded events',ylabel='Percent of paired events',ylim=(0,paired.percent_of_paired_events.max()+12))
finish(fig,'Q22_requested_vs_set','Only dated events with both nonnegative amounts; zeros retained. Missing never becomes zero. Not all hearings, defendants, cash-only/surety modes, or money collected.')
answer(22,'Answerable',f'Among {len(pair):,} paired events: lower {paired.loc["Lower","percent_of_paired_events"]:.1f}%, equal {paired.loc["Equal","percent_of_paired_events"]:.1f}%, higher {paired.loc["Higher","percent_of_paired_events"]:.1f}%. Pairs are {len(pair)/len(b):.1%} of dated bail events.',
 'Conditional observed-event answer only; repeated events can weight some cases more. One-paired-event-per-case sensitivity is provided; no actual payment information.',
 'Bail event','Events dated 2017–2024 with both nonnegative amounts recorded')
contract(22,'Lower/equal/higher bars with n and pair coverage','Bail event','Bail date 2017–2024','Paired events in each comparison class','All valid paired events','Missing distinct from recorded zero','None for the explicitly conditional question')

## Q23. How complete are the race, offense severity, judge, and bail-amount fields, and how does completeness differ across counties?

**Scope: Data quality and reliability.** Cape-only completeness is directly observable. Cross-county differences cannot be calculated without importing other-county numerical results, which this notebook does not do. Missing, explicit Unknown, unmapped, invalid, and recorded zero are separate quality states.

In [ ]:
table('Q23_field_blank_coverage',coverage,rows=35)
values=[]
for name,frame in [('Full filing extract',f_all),('Disposition rows in window',d),('Bail events in window',b)]:
    for col in ['race_code','sex_code']:
        for value,count in frame[col].fillna('Missing').value_counts().items():values.append({'population':name,'field':col,'value':value,'rows':int(count)})
    if 'source_class' in frame:
        for value,count in frame.source_class.value_counts().items():values.append({'population':name,'field':'source_class','value':value,'rows':int(count)})
table('Q23_Missing_Unknown_unmapped_values',pd.DataFrame(values),rows=50)
fig,ax=barh(coverage.set_index(coverage.population+' / '+coverage.field).missing_pct,'Cape field blank coverage','Missing (%)')
ax.axvline(20,color=COLORS[3],ls='--',label='20% threshold');ax.legend()
finish(fig,'Q23_field_coverage','Field blank rates by named population. Explicit U, unmapped codes and invalid dates are separately audited, not merged with Missing or filled.')
table('Q23_bail_amount_missing_and_zero',pd.Series({'events':len(b),'requested_missing':int(b.requested.isna().sum()),
 'set_missing':int(b.set_amount.isna().sum()),'requested_zero':int(b.requested.eq(0).sum()),'set_zero':int(b.set_amount.eq(0).sum()),
 'pairs':len(pair)},name='events'))
answer(23,'Partial','Cape field completeness, date quality, case-severity incompleteness, sentencing gaps, Missing/Unknown/unmapped codes, and zero-versus-missing amounts are supplied.',
 'No other-county numerical inputs, so no cross-county completeness difference is reported. Case-level and source-row denominators differ and must be matched before comparison.',
 'Field × stated population','All records in each named extract/window and eligible case population')
contract(23,'Coverage bars plus explicit quality-state tables','Field × population','Named actual/proxy period','Blank/Unknown/unmapped/invalid separately','Relevant records for the field','Missing ≠ Unknown ≠ invalid ≠ zero','Comparable other-county coverage to answer comparison component')

## Common-goal assessment

**Q22 is immediately reproducible as a conditional paired-event chart** if each county supplies request/set amounts and dates. Q23’s within-county coverage component is also useful before any pooled result. Case outcome/PTP/GPL/trial-marker charts can be compared only after agreeing the case/docket rule, criminal severity scope, outcome hierarchy, and missingness handling.

**Do not combine Cape proxy-year charts with Berkshire filing-year charts.** Cape needs an actual filing date for Q1’s filing companion, Q2’s annual numerator, Q13, and a uniform-date Q21. Q10’s sparse sentence fields prevent a comparable three-way sentence distribution. Exact unsupported questions and additional inputs are listed below; no outside numbers fill those gaps.

In [ ]:
QUESTION_CONFIG={1: {'scope': 'Volume and scope', 'question': 'How many cases were referred to the prosecutor each year, and how has that changed over the study period?'}, 2: {'scope': 'Volume and scope', 'question': 'What is the per-capita case rate (cases per 10,000 residents), so counties of different sizes can be compared?'}, 3: {'scope': 'Volume and scope', 'question': 'What share of cases are felonies vs. misdemeanors, and what are the 5 most common offense categories?'}, 4: {'scope': 'Charging decisions', 'question': 'What percentage of referred cases does the prosecutor decline to charge?'}, 5: {'scope': 'Charging decisions', 'question': 'How often is the filed charge different from the arrest/referral charge (upgraded, downgraded, unchanged)?'}, 6: {'scope': 'Charging decisions', 'question': 'What share of cases are diverted to alternative programs instead of prosecuted?'}, 7: {'scope': 'Case outcomes', 'question': 'Of charged cases, what percentage end in dismissal, plea, trial conviction, or acquittal?'}, 8: {'scope': 'Case outcomes', 'question': 'What is the plea rate, and how often do pleas involve a reduced charge?'}, 9: {'scope': 'Case outcomes', 'question': 'What share of cases go to trial, and what is the conviction rate at trial?'}, 10: {'scope': 'Sentencing', 'question': 'What share of convictions result in incarceration vs. probation vs. fines/other?'}, 11: {'scope': 'Sentencing', 'question': 'What is the median sentence length by offense category?'}, 12: {'scope': 'Pretrial and timing', 'question': 'What share of defendants are detained pretrial, and how often is cash bail set?'}, 13: {'scope': 'Pretrial and timing', 'question': 'What is the median time from filing to disposition, and does it differ by case type?'}, 14: {'scope': 'Equity', 'question': 'Do declination, dismissal, and incarceration rates differ by defendant race/ethnicity?'}, 15: {'scope': 'Equity', 'question': 'Do those outcomes differ by sex and age group?'}, 16: {'scope': 'Equity', 'question': 'Do racial disparities remain when you compare within the same offense category and prior-record level?'}, 17: {'scope': 'Trends and context', 'question': 'Did any of the above metrics shift noticeably around a policy change, new DA term, or 2020?'}, 18: {'scope': 'Trends and context', 'question': 'How does this county rank against the state average and the other counties on the team’s core metrics (rates from Q4, Q7, Q10, Q14)?'}, 19: {'scope': 'Case complexity', 'question': 'How does the number of recorded charges per case relate to the time from offense to disposition?'}, 20: {'scope': 'Court and agency patterns', 'question': 'Do dismissal, conviction, and diversion rates differ across courts when comparing cases of the same offense severity?'}, 21: {'scope': 'Court and agency patterns', 'question': 'Which agencies account for the largest share of filed charges, and how does their offense mix differ?'}, 22: {'scope': 'Bail decisions', 'question': 'When both amounts are recorded, how often is bail set lower than, equal to, or higher than the amount requested?'}, 23: {'scope': 'Data quality and reliability', 'question': 'How complete are the race, offense severity, judge, and bail-amount fields, and how does completeness differ across counties?'}}
matrix=pd.DataFrame([ANSWERS[i] for i in range(1,24)]).set_index('question')
matrix.insert(0,'scope',[QUESTION_CONFIG[i]['scope'] for i in matrix.index])
matrix.insert(1,'objective_question',[QUESTION_CONFIG[i]['question'] for i in matrix.index])
table('all_23_objectives_answer_matrix',matrix,rows=23)
table('uniform_visualization_contracts',pd.DataFrame(CONTRACTS).set_index('question'),rows=23)
fig,ax=barh(matrix.status.value_counts(),'Answerability of the literal 23 Objectives','Questions')
finish(fig,'objective_answerability_summary','Partial results carry the stated restrictions; no external numbers or guessed filing dates are supplied.')
assert set(ANSWERS)==set(range(1,24))
assert int(outcomes.case_units.sum())==len(dc)
assert int(paired.events.sum())==len(pair)
text='# Cape and Islands: answers to the 23 Objectives\n\nOnly three local Cape workbooks are used. No filing date is supplied; ID-year is an unverified proxy. Missing is separate from Unknown/unmapped/invalid/zero.\n\n'
for number,row in matrix.iterrows():
    text+=f'## Q{number}. {row["objective_question"]}\n\n**{row["status"]}.** {row["answer"]}\n\nLimit: {row["limitation_or_missing_input"]}\n\n'
(OUT/'answers_to_23_objectives.md').write_text(text,encoding='utf8')

assert all(hashlib.sha256(path.read_bytes()).hexdigest()==hashes[name] for name,path in paths.items())
assert fc_all.index.is_unique and dc.index.is_unique
assert first.case_id.is_unique and first_link.case_id.is_unique
manifest={'inputs':{k:{'file':v.name,'sha256':hashes[k]} for k,v in paths.items()},
 'external_numerical_inputs':[],'filing_date_present':False,'case_ID_year_is_unverified_proxy':True,
 'source_files_unchanged':True,'identical_rows_retained':True,'quarantine_rows':len(quarantine),
 'figures':FIGURES,'tables':list(TABLES),'versions':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__,'matplotlib':plt.matplotlib.__version__}}
(OUT/'analysis_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf8')
(OUT/'computed_findings.md').write_text('# Cape and Islands: computed findings\n\n'+'\n\n'.join('- '+s for s in FINDINGS),encoding='utf8')
print(f'Completed: {len(FIGURES)} figures, {len(TABLES)} tables. Original hashes unchanged; no external numerical inputs.')